## Conclusión y Próximos Pasos

Has aprendido a:

1. ✅ **Parsear datos esqueletales en XML** - Extraer información de articulaciones y dedos
2. ✅ **Visualizar esqueletos en 3D y 2D** - Entender la geometría de los gestos
3. ✅ **Crear animaciones y videos** - Renderizar secuencias de movimiento
4. ✅ **Analizar características de gestos** - Extraer patrones temporales
5. ✅ **Procesar múltiples gestos** - Automatizar la creación de videos

### Cómo crear videos como en Kaggle:

Usa el script `reconstruct_skeleton_video.py` desde la terminal:

```bash
# Video único
python src/reconstruct_skeleton_video.py \
  --gesture_path "data/dataset_skeletal_hand_gesture/skeletal/00/test_gesture/02_l/00" \
  --output skeleton_videos/letra_L.mp4 \
  --label "Letra L"

# Batch completo
python src/reconstruct_skeleton_video.py \
  --batch \
  --dataset_path "data/dataset_skeletal_hand_gesture/skeletal" \
  --output_dir skeleton_videos \
  --user_ids 0 1 2 \
  --gesture_types test_gesture
```

O usa los ejemplos del script `run_examples.py`.

In [ ]:
class SkeletonVisualizer:
    """Visualiza esqueletos de mano en 2D y 3D."""
    
    FINGER_NAMES = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
    FINGER_COLORS = {
        'Thumb': '#FF0000',   # Rojo
        'Index': '#00FF00',   # Verde
        'Middle': '#0000FF',  # Azul
        'Ring': '#FFFF00',    # Amarillo
        'Pinky': '#FF00FF'    # Magenta
    }
    
    @staticmethod
    def plot_3d_skeleton(frame_data: Dict, title: str = "Hand Skeleton 3D"):
        """Dibuja el esqueleto en 3D."""
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Dibujar centro de la mano
        if frame_data.get('hand_center'):
            center = frame_data['hand_center']
            ax.scatter([center[0]], [center[1]], [center[2]], 
                      c='black', s=100, marker='o', label='Hand Center')
        
        # Dibujar dedos
        for finger_idx, finger_name in enumerate(SkeletonVisualizer.FINGER_NAMES):
            finger_data = frame_data.get('fingers', {}).get(finger_name, {})
            
            if not finger_data:
                continue
            
            # Obtener articulaciones en orden
            joints = []
            
            # Comenzar desde MCP
            if 'mcpPosition' in finger_data:
                joints.append(finger_data['mcpPosition'])
            if 'pipPosition' in finger_data:
                joints.append(finger_data['pipPosition'])
            if 'dipPosition' in finger_data:
                joints.append(finger_data['dipPosition'])
            if 'TipPosition' in finger_data:
                joints.append(finger_data['TipPosition'])
            
            if joints:
                joints = np.array(joints)
                
                # Dibujar puntos
                ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2],
                          c=SkeletonVisualizer.FINGER_COLORS[finger_name], 
                          s=50, label=finger_name)
                
                # Dibujar líneas
                ax.plot(joints[:, 0], joints[:, 1], joints[:, 2],
                       c=SkeletonVisualizer.FINGER_COLORS[finger_name], 
                       linewidth=2, alpha=0.7)
        
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def draw_skeleton_2d(frame_data: Dict, canvas_size: Tuple[int, int] = (800, 600)) -> np.ndarray:
        """
        Dibuja el esqueleto en 2D usando proyección ortográfica.
        
        Returns:
            Array con imagen del esqueleto dibujado
        """
        canvas = np.ones((*canvas_size[::-1], 3), dtype=np.uint8) * 255
        
        if not frame_data.get('hand_center'):
            return canvas
        
        # Proyectar puntos 3D a 2D
        def project_3d_to_2d(point_3d):
            if len(point_3d) < 3:
                return (int(canvas_size[0]/2), int(canvas_size[1]/2))
            
            x, y, z = point_3d[:3]
            
            # Escalar
            scale = (canvas_size[0] - 200) / 400
            
            px = int(canvas_size[0]/2 + x * scale)
            py = int(canvas_size[1]/2 - y * scale)
            
            px = max(0, min(canvas_size[0]-1, px))
            py = max(0, min(canvas_size[1]-1, py))
            
            return (px, py)
        
        # Dibujar centro
        center_2d = project_3d_to_2d(frame_data['hand_center'])
        cv2.circle(canvas, center_2d, 6, (0, 0, 0), -1)
        
        # Dibujar dedos
        for finger_name in SkeletonVisualizer.FINGER_NAMES:
            finger_data = frame_data.get('fingers', {}).get(finger_name, {})
            
            if not finger_data:
                continue
            
            # Color BGR
            color_map = {
                'Thumb': (0, 0, 255),      # Rojo
                'Index': (0, 255, 0),      # Verde
                'Middle': (255, 0, 0),     # Azul
                'Ring': (0, 255, 255),     # Amarillo
                'Pinky': (255, 0, 255)     # Magenta
            }
            
            color = color_map.get(finger_name, (128, 128, 128))
            
            # Obtener articulaciones
            joints = []
            for joint_name in ['mcpPosition', 'pipPosition', 'dipPosition', 'TipPosition']:
                if joint_name in finger_data:
                    joints.append(finger_data[joint_name])
            
            if joints:
                joints_2d = [project_3d_to_2d(j) for j in joints]
                
                # Dibujar líneas
                for i in range(len(joints_2d) - 1):
                    cv2.line(canvas, joints_2d[i], joints_2d[i+1], color, 2)
                
                # Dibujar círculos en articulaciones
                for pt in joints_2d:
                    cv2.circle(canvas, pt, 4, color, -1)
        
        # Agregar texto
        cv2.putText(canvas, 'Hand Skeleton 2D Projection', (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)
        
        return canvas

# Visualizar primer frame en 3D
print("Visualizando primer frame en 3D...")
SkeletonVisualizer.plot_3d_skeleton(frames[0], 
                                   title=f"Hand Skeleton - Frame {frames[0].get('frame_id', 0)}")

## Sección 4: Animar Frames Esquelatales a lo Largo del Tiempo

Creamos animaciones que muestran la evolución temporal del esqueleto de la mano.

## Sección 3: Visualizar Esqueleto de Mano en 3D

Creamos una visualización 3D del esqueleto mostrando todas las articulaciones conectadas.

In [9]:
class SkeletalDataExtractor:
    """Extrae y estructura datos de coordenadas articulares."""
    
    FINGERS = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
    JOINTS = ['TipPosition', 'dipPosition', 'pipPosition', 'mcpPosition']
    
    @staticmethod
    def extract_frame_joints(frame_data: Dict) -> np.ndarray:
        """
        Extrae todas las articulaciones de un frame en un array.
        
        Returns:
            Array de shape (25, 3) - 5 dedos * 5 articulaciones (incluyendo centro)
        """
        joints = []
        
        # Centro de la mano
        if frame_data.get('hand_center'):
            joints.append(frame_data['hand_center'])
        else:
            joints.append((0, 0, 0))
        
        # Articulaciones de dedos
        for finger_name in SkeletalDataExtractor.FINGERS:
            finger_data = frame_data.get('fingers', {}).get(finger_name, {})
            for joint_name in SkeletalDataExtractor.JOINTS:
                if joint_name in finger_data:
                    joints.append(finger_data[joint_name])
                else:
                    joints.append((0, 0, 0))
        
        return np.array(joints, dtype=np.float32)
    
    @staticmethod
    def extract_sequence_joints(frames: List[Dict]) -> np.ndarray:
        """
        Extrae todas las articulaciones de una secuencia.
        
        Returns:
            Array de shape (N_frames, 25, 3)
        """
        sequence_joints = []
        for frame in frames:
            joints = SkeletalDataExtractor.extract_frame_joints(frame)
            sequence_joints.append(joints)
        
        return np.array(sequence_joints, dtype=np.float32)
    
    @staticmethod
    def extract_finger_extension_states(frames: List[Dict]) -> np.ndarray:
        """
        Extrae estados de extensión de dedos para cada frame.
        
        Returns:
            Array de shape (N_frames, 5) con 1 si extendido, 0 si no
        """
        extension_states = []
        
        for frame in frames:
            states = []
            for finger_name in SkeletalDataExtractor.FINGERS:
                finger_data = frame.get('fingers', {}).get(finger_name, {})
                is_extended = finger_data.get('isExtended', False)
                states.append(1 if is_extended else 0)
            
            extension_states.append(states)
        
        return np.array(extension_states, dtype=np.int32)

# Extraer datos de la secuencia
joints_array = SkeletalDataExtractor.extract_sequence_joints(frames)
extension_states = SkeletalDataExtractor.extract_finger_extension_states(frames)

print(f"Array de articulaciones shape: {joints_array.shape}")
print(f"Array de extensión shape: {extension_states.shape}")
print(f"\nPrimer frame, articulaciones:")
print(f"  Centro mano: {joints_array[0, 0]}")
print(f"  Índice (4 articulaciones): ")
for i, joint in enumerate(joints_array[0, 6:10]):
    print(f"    {SkeletalDataExtractor.JOINTS[i]}: {joint}")

Array de articulaciones shape: (6, 21, 3)
Array de extensión shape: (6, 5)

Primer frame, articulaciones:
  Centro mano: [  4.970009 115.34769   27.499517]
  Índice (4 articulaciones): 
    TipPosition: [-11.0992365 110.90779   -58.7208   ]
    dipPosition: [-13.571981 114.650055 -36.79489 ]
    pipPosition: [-16.662832  116.25306     2.8324404]
    mcpPosition: [ 8.916496 48.449757  9.190244]


## Sección 2: Extraer Coordenadas de Articulaciones

Extraemos las coordenadas 3D de todas las articulaciones de los dedos y organizamos en arrays estructurados.

In [7]:
class SkeletalDataParser:
    """Parser para datos esqueletales en formato XML."""
    
    FINGER_NAMES = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
    JOINT_NAMES = ['TipPosition', 'dipPosition', 'pipPosition', 'mcpPosition']
    
    @staticmethod
    def parse_xml_frame(xml_path: str) -> Dict:
        """
        Parsea un archivo XML y extrae información esqueletal.
        
        Args:
            xml_path: Ruta al archivo XML
            
        Returns:
            Diccionario con información del esqueleto
        """
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            frame_data = {
                'frame_id': None,
                'hand_center': None,
                'fingers': {},
                'confidence': None
            }
            
            # Obtener ID del frame
            frame_elem = root.find('Frame')
            if frame_elem is not None:
                id_elem = frame_elem.find('ID')
                if id_elem is not None:
                    frame_data['frame_id'] = int(id_elem.text)
            
            # Buscar la mano (RigthImage/Right)
            right_image = root.find('.//RigthImage')
            if right_image is None:
                return frame_data
                
            hands = right_image.find('.//Hands')
            if hands is None:
                return frame_data
            
            hand = hands.find('.//Right')
            if hand is None:
                return frame_data
            
            # Extraer centro de la mano
            center_elem = hand.find('Center')
            if center_elem is not None and center_elem.find('data') is not None:
                center_data = center_elem.find('data').text.strip().split()
                frame_data['hand_center'] = tuple(float(x) for x in center_data)
            
            # Extraer confianza
            confidence_elem = hand.find('Confidence')
            if confidence_elem is not None:
                frame_data['confidence'] = float(confidence_elem.text)
            
            # Extraer información de dedos
            fingers_elem = hand.find('Fingers')
            if fingers_elem is not None:
                for finger_name in SkeletalDataParser.FINGER_NAMES:
                    finger = fingers_elem.find(finger_name)
                    if finger is not None:
                        frame_data['fingers'][finger_name] = {}
                        
                        # Extraer posiciones de articulaciones
                        for joint_name in SkeletalDataParser.JOINT_NAMES:
                            joint = finger.find(joint_name)
                            if joint is not None and joint.find('data') is not None:
                                joint_data = joint.find('data').text.strip().split()
                                frame_data['fingers'][finger_name][joint_name] = tuple(
                                    float(x) for x in joint_data
                                )
                        
                        # Extraer si está extendido
                        is_extended = finger.find('isExtended')
                        if is_extended is not None:
                            frame_data['fingers'][finger_name]['isExtended'] = (
                                is_extended.text == '1'
                            )
            
            return frame_data
            
        except Exception as e:
            print(f"Error parseando {xml_path}: {e}")
            return {}

    @staticmethod
    def load_gesture_sequence(gesture_path: str) -> List[Dict]:
        """
        Carga todos los frames de una secuencia de gesto.
        
        Args:
            gesture_path: Ruta a la carpeta con frames XML
            
        Returns:
            Lista de diccionarios con información de cada frame
        """
        xml_files = sorted([
            os.path.join(gesture_path, f) 
            for f in os.listdir(gesture_path) 
            if f.endswith('.xml')
        ])
        
        frames = []
        for xml_file in xml_files:
            frame_data = SkeletalDataParser.parse_xml_frame(xml_file)
            frames.append(frame_data)
        
        return frames

# Probar el parser
dataset_base = "../data/dataset_skeletal_hand_gesture/skeletal"
test_gesture_path = os.path.join(dataset_base, "00", "test_gesture", "02_l", "00")

print(f"Cargando frames de: {test_gesture_path}")
frames = SkeletalDataParser.load_gesture_sequence(test_gesture_path)
print(f"✓ Se cargaron {len(frames)} frames")

Cargando frames de: ../data/dataset_skeletal_hand_gesture/skeletal\00\test_gesture\02_l\00
✓ Se cargaron 6 frames


## Sección 1: Cargar y Parsear Datos Esqueletales en XML

Cargamos los archivos XML que contienen la información esqueletal de cada frame.

In [ ]:
# Importar librerías necesarias
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import pandas as pd
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

print("Librerías importadas correctamente")

Librerías importadas correctamente


# Reconstrucción de Videos de Gestos desde Datos Esqueletales

Este notebook permite reconstruir videos de gestos de mano a partir de archivos XML con información esqueletal.
Es útil para visualizar exactamente qué gesto se está realizando y generar videos similares a los de Kaggle.

**Objetivo:** Transformar datos de puntos articulares de la mano en videos visuales del esqueleto.

In [ ]:
# Visualizar varios frames en 2D
n_frames_to_show = min(4, len(frames))
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

frame_indices = np.linspace(0, len(frames)-1, n_frames_to_show, dtype=int)

for idx, frame_idx in enumerate(frame_indices):
    frame = frames[frame_idx]
    canvas = SkeletonVisualizer.draw_skeleton_2d(frame)
    
    axes[idx].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"Frame {frame_idx + 1}/{len(frames)}", fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"✓ Se visualizaron {n_frames_to_show} frames de la secuencia")

## Sección 5: Clasificación de Gestos desde Secuencias Esquelatales

Analizamos características temporales para identificar tipos de gestos.

In [ ]:
class GestureFeatureExtractor:
    """Extrae características de gestos desde datos esqueletales."""
    
    @staticmethod
    def get_finger_extension_profile(frames: List[Dict]) -> Dict[str, np.ndarray]:
        """
        Obtiene perfil de extensión de cada dedo a lo largo del tiempo.
        
        Returns:
            Diccionario con array para cada dedo (N_frames,)
        """
        fingers = ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
        profiles = {}
        
        for finger in fingers:
            extension_seq = []
            for frame in frames:
                finger_data = frame.get('fingers', {}).get(finger, {})
                is_ext = 1 if finger_data.get('isExtended', False) else 0
                extension_seq.append(is_ext)
            
            profiles[finger] = np.array(extension_seq)
        
        return profiles
    
    @staticmethod
    def compute_hand_motion(frames: List[Dict]) -> Tuple[float, float, float]:
        """
        Calcula movimiento de la mano (velocidad promedio).
        
        Returns:
            (avg_velocity_x, avg_velocity_y, avg_velocity_z)
        """
        centers = []
        for frame in frames:
            if frame.get('hand_center'):
                centers.append(frame['hand_center'])
        
        if len(centers) < 2:
            return (0, 0, 0)
        
        centers = np.array(centers)
        velocities = np.diff(centers, axis=0)
        
        return tuple(np.mean(velocities, axis=0))
    
    @staticmethod
    def compute_hand_opening_degree(frames: List[Dict]) -> np.ndarray:
        """
        Calcula el grado de apertura de la mano (basado en dedos extendidos).
        
        Returns:
            Array de shape (N_frames,) con valores de 0 a 5
        """
        opening = []
        for frame in frames:
            n_extended = sum(
                1 for finger in ['Thumb', 'Index', 'Middle', 'Ring', 'Pinky']
                if frame.get('fingers', {}).get(finger, {}).get('isExtended', False)
            )
            opening.append(n_extended)
        
        return np.array(opening)

# Extraer características
print("Extrayendo características del gesto...")

extension_profiles = GestureFeatureExtractor.get_finger_extension_profile(frames)
hand_motion = GestureFeatureExtractor.compute_hand_motion(frames)
hand_opening = GestureFeatureExtractor.compute_hand_opening_degree(frames)

print(f"\nCaracterísticas del gesto:")
print(f"  Movimiento promedio: {hand_motion}")
print(f"  Apertura de mano (min-max): {hand_opening.min()}-{hand_opening.max()}")

# Visualizar perfil de extensión
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, (finger, profile) in enumerate(extension_profiles.items()):
    ax = axes[idx]
    ax.plot(profile, linewidth=2, marker='o', markersize=4)
    ax.set_title(f"{finger} Extension", fontweight='bold')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Extended (0=No, 1=Yes)')
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)

# Subplot adicional para apertura total
ax = axes[5]
ax.plot(hand_opening, linewidth=2, marker='o', markersize=4, color='purple')
ax.set_title("Hand Opening Degree", fontweight='bold')
ax.set_xlabel('Frame')
ax.set_ylabel('Number of Extended Fingers')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Sección 6: Generar Videos Sintéticos de Gestos de Mano

Usamos los datos esquelatales para renderizar videos profesionales.

In [ ]:
class VideoReconstructor:
    """Reconstruye videos a partir de datos esqueletales."""
    
    def __init__(self, fps: int = 30, canvas_size: Tuple[int, int] = (800, 600)):
        self.fps = fps
        self.canvas_size = canvas_size
    
    def render_video(self, frames: List[Dict], output_path: str, 
                    gesture_label: str = "") -> bool:
        """
        Render video de los frames esquelatales.
        
        Args:
            frames: Lista de diccionarios con datos de frames
            output_path: Ruta del video de salida
            gesture_label: Etiqueta del gesto
            
        Returns:
            True si fue exitoso
        """
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(output_path, fourcc, self.fps, self.canvas_size)
        
        if not writer.isOpened():
            print(f"Error: No se puede crear el video {output_path}")
            return False
        
        print(f"Renderizando video ({len(frames)} frames)...")
        
        for i, frame_data in enumerate(frames):
            # Dibujar esqueleto
            canvas = SkeletonVisualizer.draw_skeleton_2d(frame_data, self.canvas_size)
            
            # Agregar información
            text = f"Frame {i+1}/{len(frames)}"
            if gesture_label:
                text += f" - {gesture_label}"
            
            cv2.putText(canvas, text, (10, 50),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)
            
            writer.write(canvas)
            
            if (i + 1) % max(1, len(frames)//5) == 0:
                print(f"  Procesado {i+1}/{len(frames)} frames...")
        
        writer.release()
        print(f"✓ Video guardado en: {output_path}")
        return True

# Crear y renderizar video
reconstructor = VideoReconstructor(fps=30)
output_video = "../skeleton_videos/letra_L_demo.mp4"
os.makedirs(os.path.dirname(output_video), exist_ok=True)

success = reconstructor.render_video(
    frames,
    output_video,
    gesture_label="Letra L (Usuario 0)"
)

if success:
    print(f"\n✓ Video creado exitosamente")

In [ ]:
class BatchGestureProcessor:
    """Procesa múltiples gestos en batch."""
    
    def __init__(self, dataset_base: str, output_base: str):
        self.dataset_base = dataset_base
        self.output_base = output_base
        self.reconstructor = VideoReconstructor(fps=30)
    
    def get_gesture_list(self, user_id: int = 0, gesture_type: str = 'test_gesture') -> List[str]:
        """Obtiene lista de gestos disponibles."""
        gesture_dir = os.path.join(self.dataset_base, str(user_id).zfill(2), gesture_type)
        
        if not os.path.exists(gesture_dir):
            return []
        
        return sorted(os.listdir(gesture_dir))
    
    def process_gesture(self, user_id: int, gesture_name: str, 
                       gesture_type: str = 'test_gesture',
                       seq_id: str = '00') -> bool:
        """Procesa un gesto específico."""
        
        gesture_path = os.path.join(
            self.dataset_base, 
            str(user_id).zfill(2), 
            gesture_type, 
            gesture_name,
            seq_id
        )
        
        if not os.path.exists(gesture_path):
            return False
        
        # Cargar frames
        frames = SkeletalDataParser.load_gesture_sequence(gesture_path)
        
        if not frames:
            return False
        
        # Crear directorio de salida
        output_dir = os.path.join(
            self.output_base,
            str(user_id).zfill(2),
            gesture_type,
            gesture_name
        )
        os.makedirs(output_dir, exist_ok=True)
        
        # Guardar video
        output_video = os.path.join(output_dir, f"{gesture_name}_seq{seq_id}.mp4")
        
        return self.reconstructor.render_video(
            frames,
            output_video,
            gesture_label=f"{gesture_name} - User {user_id} - Seq {seq_id}"
        )

# Crear procesador batch
processor = BatchGestureProcessor(dataset_base, "../skeleton_videos")

# Obtener lista de gestos disponibles
gestures = processor.get_gesture_list(user_id=0, gesture_type='test_gesture')

print(f"Gestos disponibles para Usuario 0:")
for idx, gesture in enumerate(gestures, 1):
    print(f"  {idx}. {gesture}")

# Procesar algunos gestos ejemplo
print(f"\nProcesando múltiples gestos...")
processed_count = 0

for gesture_name in gestures[:3]:  # Procesar los primeros 3 gestos
    print(f"\nProcesando: {gesture_name}")
    if processor.process_gesture(user_id=0, gesture_name=gesture_name, seq_id='00'):
        processed_count += 1

print(f"\n✓ {processed_count} gestos procesados exitosamente")